In [1]:
import math
import numpy as np

import ipywidgets as widgets
from IPython.display import display, HTML

import matplotlib.pyplot as plt


# Production Profit Optimizer

### Mathematical Optimization of Production and Pricing Under Uncertain Demand

## 1. Project Overview

Small businesses often need to decide how many units of a product to order and what selling price to set. These decisions are important because production costs must be paid even when some products remain unsold.

This project develops a mathematical model to help businesses explore production and pricing decisions under a limited budget and uncertain customer demand.

The model is designed for businesses that purchase finished products from external manufacturers and then sell them to customers.

A protein candy business is used as the main practical example. However, the mathematical approach can also be applied to other products by changing the input parameters.



## 2. Problem Statement and Objectives

The main goal of this project is to determine the production quantity and selling price that maximize expected profit while respecting business constraints.

The model considers the following factors:

- Available production budget.
- Fixed and variable production costs.
- Minimum order quantity (MOQ).
- A range of possible selling prices.
- Customer demand estimates at two different prices.
- Uncertainty in actual customer demand.

### Research Questions

The project aims to answer four main questions:

1. What production quantities are feasible given the budget and MOQ?
2. Which combination of production quantity and selling price maximizes profit under expected demand?
3. How does uncertainty in customer demand affect expected profit and the probability of financial loss?
4. How does the available production budget affect the maximum achievable profit, and at what point does increasing the budget stop improving profitability?

### Project Approach

First, a deterministic mathematical model will be developed to analyze production costs, demand, revenue, and profit.

Next, mathematical optimization will be used to identify the most profitable feasible decisions.

Finally, Monte Carlo simulations will be performed to estimate expected profit and financial risk under uncertain demand.

The results will be validated through mathematical checks, numerical experiments, and visualizations.



## 3. Assumptions and Scope

To keep the mathematical model understandable and manageable, the following assumptions are made:

1. **Single product:** The model analyzes one product at a time. All quantities refer to finished units ready for sale.

2. **External manufacturing:** The business purchases finished products from an external manufacturer rather than producing them itself.

3. **Constant unit cost:** The production cost per unit remains constant within the analyzed order. Quantity discounts are not considered.

4. **Fixed costs:** Fixed costs do not change with the ordered quantity.

5. **Budget and MOQ:** Production decisions must satisfy the available budget and the manufacturer's minimum order quantity.

6. **Price-dependent demand:** Expected customer demand is approximated by a linear function derived from two price-demand estimates provided by the user.

7. **Single sales period:** All sales are evaluated within one period. Unsold products generate no revenue or residual value during that period.

8. **Uncertain demand:** Actual demand may differ from expected demand. The variability is represented by an explicitly stated probability model.

9. **Synthetic data:** The main experiments use illustrative input values rather than verified historical sales data.

10. **Simplified financial model:** Taxes, financing costs, storage costs, and other expenses not explicitly included in the input parameters are outside the model.

### Scope and Limitations

The project focuses on mathematical modeling, constrained profit optimization, numerical experiments, and Monte Carlo simulation.

It does not attempt to forecast actual market demand, predict future business performance, or provide guaranteed financial recommendations.

The results are conditional on the input data and modeling assumptions. Different assumptions may lead to different optimal decisions.



## 4. Mathematical Model

### 4.1. Input Parameters

The model begins with four basic production parameters: the available budget, fixed costs, unit production cost, and minimum order quantity (MOQ).

The values below represent a synthetic example of a small business purchasing finished products from an external manufacturer.

All monetary values are expressed in euros (EUR).


In [2]:

# Basic production parameters
budget = 5000
fixed_cost = 500
unit_cost = 1.50
moq = 1000

print(f"Available budget: {budget} EUR")
print(f"Fixed costs: {fixed_cost} EUR")
print(f"Unit cost: {unit_cost} EUR")
print(f"Minimum order quantity: {moq} units")


Available budget: 5000 EUR
Fixed costs: 500 EUR
Unit cost: 1.5 EUR
Minimum order quantity: 1000 units



### 4.2. Production Cost Model

Total production cost consists of fixed costs and variable costs.

The cost function is:

$$
C(q) = F + cq
$$

where:

- $C(q)$ is the total cost in EUR.
- $F$ is the fixed cost in EUR.
- $c$ is the cost per unit in EUR.
- $q$ is the ordered quantity.

The model assumes that fixed costs and unit costs remain constant within the analyzed order.


In [3]:
# Example: verify the production cost formula
example_quantity = 3000

example_total_cost = fixed_cost + unit_cost * example_quantity

print(f"Cost for {example_quantity} units: {example_total_cost:.2f} EUR")

Cost for 3000 units: 5000.00 EUR



### 4.3. Budget and Minimum Order Quantity Constraints

The business cannot order an unlimited number of units. The order must satisfy two constraints.

**Budget constraint:**

$$
F + cq \leq B
$$

**Minimum order quantity constraint:**

$$
q \geq MOQ
$$

where:

- $B$ is the available budget.
- $F$ is the fixed cost.
- $c$ is the unit production cost.
- $q$ is the ordered quantity.
- $MOQ$ is the manufacturer's minimum order quantity.

The maximum affordable quantity is:

$$
q_{\max} = \left\lfloor \frac{B-F}{c} \right\rfloor
$$

The floor operation ensures that the quantity is a whole number.

A feasible order exists only if:

$$
q_{\max} \geq MOQ
$$

For the example parameters, the feasible quantity range is from 1,000 to 3,000 units.

The maximum affordable quantity is not necessarily the most profitable quantity. The optimal order will be determined later using the profit model.


In [4]:
max_quantity = int((budget - fixed_cost) // unit_cost)

if max_quantity >= moq:
    print(max_quantity)
else:
    print("Insufficient budget to meet MOQ.")

3000


The cost formula is implemented as a reusable Python function, allowing the model to calculate production costs for different order quantities.

In [5]:
def calculate_total_cost(quantity, fixed_cost, unit_cost):
    total_cost = fixed_cost + unit_cost * quantity

    return total_cost

assert calculate_total_cost(1000, fixed_cost, unit_cost) == 2000
assert calculate_total_cost(3000, fixed_cost, unit_cost) == 5000

print("All production cost tests passed successfully!")

All production cost tests passed successfully!



### 4.4. Customer Demand Model

Customer demand is influenced by the selling price. The model assumes a linear relationship between price and expected demand within the analyzed price range.

The demand function is:

$$
D(p) = a + bp
$$

where:
- $D(p)$ is the expected customer demand in units.
- $p$ is the selling price per unit in EUR.
- $a$ is the intercept of the demand function.
- $b$ is the change in expected demand for a 1 EUR increase in price.

The coefficients are calculated from two user-provided price-demand estimates:

$$
b = \frac{D_2-D_1}{p_2-p_1}
$$

$$
a = D_1-bp_1
$$

The two prices must be different to avoid division by zero.

For the synthetic example:
- At 3 EUR, expected demand is 2,000 units.
- At 5 EUR, expected demand is 1,000 units.

Therefore:

$$
b = \frac{1000-2000}{5-3} = -500
$$

$$
a = 2000-(-500 \cdot 3) = 3500
$$

The resulting demand function is:

$$
D(p) = 3500-500p
$$

This relationship is an assumption based on the provided estimates, not a guarantee of actual sales. The model is applied only within the price range defined by the two estimates.


In [6]:
price_1 = 3.0
demand_1 = 2000

price_2 = 5.0
demand_2 = 1000

if price_1 <= 0 or price_2 <= 0:
    raise ValueError("The selling prices must be positive.")

if demand_1 < 0 or demand_2 < 0:
    raise ValueError("Expected demand cannot be negative.")

if price_1 == price_2:
    raise ValueError("The two prices must be different.")

b = (demand_2 - demand_1) / (price_2 - price_1)

if b > 0:
    raise ValueError("Expected demand must not increase as price increases.")

a = demand_1 - (b * price_1)

print(f"Demand intercept (a): {a:.2f}")
print(f"Demand slope (b): {b:.2f}")


Demand intercept (a): 3500.00
Demand slope (b): -500.00


In [7]:
def calculate_demand(price, a, b):
    demand = a + (b * price)

    return demand

# Validate the customer demand model
assert calculate_demand(price_1, a, b) == demand_1
assert calculate_demand(price_2, a, b) == demand_2

print("Customer demand model tests passed successfully!")

Customer demand model tests passed successfully!



#### Input Validation and Model Assumptions

The input parameters are validated before calculating the demand function.

The following conditions must be satisfied:

- Both selling prices must be positive.
- Expected demand values must be non-negative.
- The two selling prices must be different.
- Expected demand must not increase as the selling price increases.

For the linear demand function:

$$
D(p) = a + bp
$$

the final condition requires:

$$
b \leq 0
$$

Invalid inputs raise a `ValueError` with an explanatory message.

These restrictions ensure that the input data is consistent with the assumptions of the selected demand model.

The model is valid only within the price interval defined by the two input estimates. Predictions outside this interval are not considered in the optimization.



### 4.5. Revenue and Profit Model

The number of units sold cannot exceed either the available stock or customer demand.

Expected sales under the deterministic demand model are:

$$
S(q,p) = \min(q,D(p))
$$

where:
- $q$ is the ordered quantity.
- $p$ is the selling price per unit.
- $D(p)$ is expected customer demand.
- $S(q,p)$ is the expected quantity sold under the deterministic demand assumption.

Revenue is calculated as:

$$
R(q,p) = p \cdot S(q,p)
$$

Total profit is revenue minus production costs:

$$
P(q,p) = R(q,p) - C(q)
$$

Combining the formulas:

$$
P(q,p) = p \cdot \min(q,D(p)) - (F+cq)
$$

The model assumes that unsold units do not generate revenue during the analyzed period, while their production costs are still incurred.

#### Example

For an order of 1,800 units at a selling price of 4 EUR:

- Expected demand: 1,500 units.
- Expected sales: 1,500 units.
- Revenue: 6,000 EUR.
- Total costs: 3,200 EUR.
- Expected profit: 2,800 EUR.

This profit is calculated under deterministic expected demand. Demand uncertainty will be introduced later.


In [8]:
def calculate_sales(quantity, price, a, b):
    demand = calculate_demand(price, a, b)
    sales = min(quantity, demand)

    return sales

assert calculate_sales(1000, 4, a, b) == 1000
assert calculate_sales(1800, 4, a, b) == 1500

In [9]:
def calculate_revenue(quantity, price, a, b):
    sales = calculate_sales(quantity, price, a, b)
    revenue = sales * price 

    return revenue

assert calculate_revenue(1800, 4, a, b) == 6000

In [10]:
def calculate_profit(quantity, price, fixed_cost, unit_cost, a, b):
    revenue = calculate_revenue(quantity, price, a, b)
    total_cost = calculate_total_cost(quantity, fixed_cost, unit_cost)
    profit = revenue - total_cost

    return profit

assert calculate_profit(1800, 4, fixed_cost, unit_cost, a, b) == 2800
assert calculate_profit(2000, 4, fixed_cost, unit_cost, a, b) == 2500

print("Profit model tests passed successfully!")

Profit model tests passed successfully!



### 4.6. Deterministic Profit Optimization

The objective is to find the selling price and order quantity that maximize profit under the assumed demand model.

The objective function is:

$$
\max_{q,p} P(q,p)
= p \cdot \min(q,D(p)) - (F+cq)
$$

subject to:

$$
MOQ \leq q \leq q_{\max}, \quad q \in \mathbb{Z}
$$

$$
p_{\min} \leq p \leq p_{\max}
$$

where the price limits are determined by the two user-provided price-demand estimates.

#### Optimization Method: Grid Search

The algorithm evaluates selling prices from the minimum to the maximum input price in increments of 0.01 EUR.

For each candidate price:

1. Calculate the expected customer demand.
2. Determine the most profitable feasible integer order quantity.
3. Calculate the corresponding profit.
4. Compare the result with the best solution found so far.

For a fixed price above the unit cost, the optimal quantity can be found by comparing the feasible integer quantities immediately below and above expected demand, after applying the MOQ and budget limits.

If the selling price does not exceed the unit cost, the model chooses MOQ, assuming an order is mandatory.

The model assumes deterministic demand, constant unit costs, and no revenue from unsold units during the analyzed period.

The best result is selected from the evaluated price grid. Therefore, the result is optimal for the tested prices, but it is not yet an analytical proof of a continuous global maximum.


In [11]:
def calculate_optimal_quantity(price, a, b, moq, max_quantity, fixed_cost, unit_cost):
    demand = calculate_demand(price, a, b)

    if moq > max_quantity:
        raise ValueError("The budget is insufficient to meet the minimum order quantity (MOQ).")

    if price <= unit_cost:
        return moq

    lower_quantity = math.floor(demand)
    upper_quantity = math.ceil(demand)

    lower_quantity = max(moq, min(lower_quantity, max_quantity))
    upper_quantity = max(moq, min(upper_quantity, max_quantity))

    lower_profit = calculate_profit(lower_quantity, price, fixed_cost, unit_cost, a, b)
    upper_profit = calculate_profit(upper_quantity, price, fixed_cost, unit_cost, a, b)

    if lower_profit < upper_profit:
        best_quantity = upper_quantity
    else:
        best_quantity = lower_quantity

    return best_quantity

assert calculate_optimal_quantity(4, a, b, moq, max_quantity, fixed_cost, unit_cost) == 1500
assert calculate_optimal_quantity(5, a, b, moq, max_quantity, fixed_cost, unit_cost) == 1000
assert calculate_optimal_quantity(3.9988, a, b, moq, max_quantity, fixed_cost, unit_cost) == 1501
assert calculate_optimal_quantity(1.20, a, b, moq, max_quantity, fixed_cost, unit_cost) == 1000

print("Optimal quantity tests passed successfully!")

Optimal quantity tests passed successfully!


In [12]:
best_profit = float("-inf")
best_price = None
best_quantity = None

min_price = min(price_1, price_2)
max_price = max(price_1, price_2)

for price_cents in range(round(min_price * 100), round(max_price * 100) + 1):
    price = price_cents / 100

    optimal_quantity = calculate_optimal_quantity(price, a, b, moq, max_quantity, fixed_cost, unit_cost)
    profit = calculate_profit(optimal_quantity, price, fixed_cost, unit_cost, a, b)

    if profit > best_profit:
        best_profit = profit
        best_price = price
        best_quantity = optimal_quantity

print(f"Optimal selling price: {best_price:.2f} EUR")
print(f"Optimal order quantity: {best_quantity} units")
print(f"Maximum profit: {best_profit:.2f} EUR")

assert best_quantity >= moq
assert best_quantity <= max_quantity
assert calculate_total_cost(best_quantity, fixed_cost, unit_cost) <= budget
assert min_price <= best_price <= max_price

Optimal selling price: 4.25 EUR
Optimal order quantity: 1375 units
Maximum profit: 3281.25 EUR



### 4.7. Analytical Verification Using Derivatives

To verify the result of the deterministic grid search, we use calculus to find the maximum of the profit function analytically.

For a selling price above the unit cost, and assuming that the optimal order quantity equals expected demand and satisfies the business constraints:

$$
q = D(p)
$$

The profit function becomes:

$$
P(p) = (p-c)D(p)-F
$$

Since expected demand is linear:

$$
D(p) = a+bp
$$

we obtain:

$$
P(p) = (p-c)(a+bp)-F
$$

Expanding the expression:

$$
P(p) = bp^2+(a-bc)p-(ac+F)
$$

For our example:

$$
P(p) = -500p^2+4250p-5750
$$

**First derivative:**

$$
P'(p) = -1000p+4250
$$

Setting the first derivative to zero:

$$
-1000p+4250 = 0
$$

$$
p^* = 4.25
$$

**Second derivative:**

$$
P''(p) = -1000 < 0
$$

Therefore, the critical point is the global maximum of this quadratic profit function.

The corresponding expected demand is:

$$
q^* = D(4.25) = 1375
$$

This quantity satisfies the minimum order quantity and budget constraints. The optimal price is also within the allowed price range.

The resulting maximum expected profit is:

$$
P^* = 3281.25 \text{ EUR}
$$

The analytical result matches the deterministic grid search result for the example inputs.

**Limitation:** This analytical approach assumes that ordering the expected demand is optimal and feasible. If the price, budget, or MOQ constraints are binding, the constrained optimum may differ. The grid search remains the primary optimization method.


In [13]:
quadratic_a = b
quadratic_b = a - (b * unit_cost)
analytical_price = -(quadratic_b / (2*quadratic_a))

analytical_quantity = calculate_demand(analytical_price, a, b)

second_derivative = 2 * quadratic_a
assert second_derivative < 0

print(analytical_quantity)
print(analytical_price)

assert abs(analytical_price - best_price) < 1e-9
assert abs(analytical_quantity - best_quantity) < 1e-9

1375.0
4.25



### 4.8. Stochastic Demand Model and Monte Carlo Simulation

#### Stochastic Demand

Unlike the deterministic model, which assumes a fixed expected demand for each selling price, the stochastic model accounts for uncertainty in customer demand.

Expected demand is still calculated using the linear function:

$$
D(p) = a + bp
$$

We assume that actual customer demand approximately follows a normal distribution:

$$
X_p \sim \mathcal{N}(D(p), \sigma(p)^2)
$$

where:

- $X_p$ is random customer demand at selling price $p$.
- $D(p)$ is the expected demand.
- $\sigma(p)$ is the standard deviation of demand.

The standard deviation is modeled as:

$$
\sigma(p) = r \cdot D(p)
$$

where $r$ is the uncertainty rate.

For the initial scenario, we assume $r = 0.20$, meaning that the standard deviation equals 20% of expected demand.

This percentage is a modeling assumption, not a value estimated from historical sales data. Users can explore different uncertainty rates in sensitivity analysis.

Since the number of products demanded cannot be negative or fractional, simulated demand is rounded to the nearest integer and limited to a minimum of zero.

#### Stochastic Profit

For an order quantity $q$, selling price $p$, and randomly generated demand $X_p$, actual sales are:

$$
S(q,p,X_p) = \min(q,X_p)
$$

The simulated profit is:

$$
P(q,p,X_p) = p \cdot \min(q,X_p) - (F+cq)
$$

Unsold products remain a cost to the business. The model does not assume additional revenue from unsold inventory.

#### Monte Carlo Simulation

Monte Carlo simulation evaluates a business decision under uncertainty by generating many independent random demand scenarios.

For each simulation:

1. Generate random customer demand.
2. Determine the number of products sold.
3. Calculate revenue and total cost.
4. Calculate and store the simulated profit.

We use 10,000 simulations for the initial analysis.

#### Risk Analysis

The simulation results are evaluated using three metrics.

**Average Simulated Profit**

$$
\overline{P} = \frac{1}{N}\sum_{i=1}^{N}P_i
$$

This estimates the expected profit under the assumed stochastic demand model.

**Probability of Loss**

$$
\widehat{P}(\text{Loss}) =
\frac{\#\{P_i < 0\}}{N}
$$

This estimates the fraction of simulated scenarios resulting in a negative profit.

**Standard Deviation of Profit**

$$
\sigma_P =
\sqrt{\frac{1}{N}\sum_{i=1}^{N}(P_i-\overline{P})^2}
$$

This measures how widely simulated profits vary around their average.

#### Limitations

The results depend on the assumed demand model, uncertainty rate, and other input parameters. They are estimates rather than guarantees of future sales or profitability.

Rounding and clipping simulated demand to non-negative integers also modify the theoretical normal distribution.

This stage evaluates a selected price and order quantity. Optimization under uncertainty is presented in Section 4.9.


In [14]:
uncertainty_rate = 0.20

def calculate_demand_std(price, a, b, uncertainty_rate):
    demand = calculate_demand(price, a, b)
    std = uncertainty_rate * demand

    return std

assert calculate_demand_std(4.25, a, b, uncertainty_rate) == 275.0
assert calculate_demand_std(5, a, b, uncertainty_rate) == 200.0


In [15]:
def generate_random_demand(price, a, b, uncertainty_rate):
    demand = calculate_demand(price, a, b)
    demand_std = calculate_demand_std(price, a, b, uncertainty_rate)
    random_demand = round(np.random.normal(loc=demand, scale=demand_std))

    return max(0, random_demand)


random_demand = generate_random_demand(4.25, a, b, uncertainty_rate)

assert isinstance(random_demand, int)
assert random_demand >= 0

print(f"Generated demand: {random_demand} units")

Generated demand: 1429 units


In [16]:
def calculate_stochastic_profit(quantity, price, fixed_cost, unit_cost, a, b, uncertainty_rate):
    random_demand = generate_random_demand(price, a, b, uncertainty_rate)
    total_cost = calculate_total_cost(quantity, fixed_cost, unit_cost)

    stochastic_profit = price * (min(quantity,random_demand)) - total_cost

    return stochastic_profit

simulated_profit = calculate_stochastic_profit(1375, 4.25, fixed_cost, unit_cost, a, b, uncertainty_rate)
assert simulated_profit <= calculate_profit(1375, 4.25, fixed_cost, unit_cost, a, b)

In [17]:
def monte_carlo_simulation(quantity, price, num_simulations, fixed_cost, unit_cost, a, b, uncertainty_rate):
    profits = []
    for n in range(num_simulations):
        stochastic_profit = calculate_stochastic_profit(quantity, price, fixed_cost, unit_cost, a, b, uncertainty_rate)
        profits.append(stochastic_profit)

    return profits

profits = monte_carlo_simulation(1375, 4.25, 10000, fixed_cost, unit_cost, a, b, uncertainty_rate)

In [18]:
def analyze_simulation(profits):

    average_profit = sum(profits) / len(profits)

    loss_count = 0
    for p in profits:
        if p < 0:
            loss_count += 1

    loss_probability = loss_count / len(profits)

    profit_std = np.std(profits)

    return average_profit, loss_probability, profit_std


In [19]:
average_profit, loss_probability, profit_std = analyze_simulation(profits)

print(f"Number of simulations: {len(profits)}")
print(f"Average simulated profit: {average_profit:.2f} EUR")
print(f"Probability of loss: {loss_probability * 100:.2f}%")
print(f"Standard deviation of profit: {profit_std:.2f} EUR")

Number of simulations: 10000
Average simulated profit: 2812.64 EUR
Probability of loss: 0.21%
Standard deviation of profit: 681.87 EUR


In [20]:
profits_1 = monte_carlo_simulation(1375, 4.25, 10000, fixed_cost, unit_cost, a, b, uncertainty_rate)
print(analyze_simulation(profits_1))

profits_2 = monte_carlo_simulation(1600, 4.25, 10000, fixed_cost, unit_cost, a, b, uncertainty_rate)
print(analyze_simulation(profits_2))

(2822.571725, 0.0021, np.float64(672.7417021779418))
(2805.395925, 0.0057, np.float64(965.7560504468219))



### 4.9. Stochastic Profit Optimization

#### Objective

The deterministic optimization model assumes that customer demand is known with certainty. In practice, actual demand may differ from its expected value, which affects the number of units sold and the final profit.

The objective of stochastic optimization is to find the selling price and production quantity that maximize the **average simulated profit** under uncertain customer demand.

The optimization problem is:

$$
\max_{p,q} \mathbb{E}[P(q,p)]
$$

where:

- $p$ is the selling price per unit.
- $q$ is the production quantity.
- $P(q,p)$ is the profit under uncertain demand.
- $\mathbb{E}[P(q,p)]$ is the expected profit, approximated using Monte Carlo simulation.

#### Common Random Numbers

To compare different pricing and production decisions fairly, the optimization uses a technique called **Common Random Numbers (CRN)**.

Instead of generating independent random demand samples for every candidate decision, the algorithm generates one set of standard normal random factors:

$$
Z_i \sim \mathcal{N}(0,1)
$$

For each selling price, the demand scenarios are calculated as:

$$
D_i(p)=\max\left(0,\operatorname{round}\left(D(p)+\sigma(p)Z_i\right)\right)
$$

where:

- $D(p)$ is the expected customer demand at price $p$.
- $\sigma(p)$ is the standard deviation of demand.
- $Z_i$ is the random factor for simulation $i$.
- $D_i(p)$ is the simulated demand in scenario $i$.

The same random factors are reused for all candidate prices and quantities. This reduces unnecessary random variation when comparing alternative decisions.

#### Profit Calculation

For each demand scenario, the actual number of units sold is limited by both production quantity and customer demand:

$$
S_i(q,p)=\min(q,D_i(p))
$$

The simulated profit is:

$$
P_i(q,p)=p\cdot S_i(q,p)-(F+cq)
$$

The average simulated profit is calculated as:

$$
\overline{P}(q,p)=\frac{1}{N}\sum_{i=1}^{N}P_i(q,p)
$$

where $N$ is the number of Monte Carlo simulations.

#### Optimization Algorithm

The stochastic optimizer uses a grid search to evaluate candidate selling prices and production quantities.

The algorithm follows these steps:

1. Generate candidate prices between the two observed selling prices.
2. Generate a shared set of standard normal random factors.
3. Calculate simulated demand for each candidate price.
4. Evaluate all feasible integer production quantities.
5. Calculate average simulated profit for every price–quantity combination.
6. Store the combination with the highest average simulated profit.
7. Verify that the selected production quantity satisfies the minimum order quantity and budget constraints.

The constraints are:

$$
q\geq MOQ
$$

$$
F+cq\leq B
$$

where $B$ is the available production budget.

#### Interpretation

Unlike deterministic optimization, stochastic optimization considers possible fluctuations in customer demand.

The optimal production quantity may differ from expected demand because producing additional units can increase sales in high-demand scenarios, while unsold units still incur production costs.

The stochastic optimizer therefore balances the opportunity to sell more units against the risk of producing excess inventory.

The returned values are:

- **Best selling price:** the price with the highest estimated average profit among the evaluated combinations.
- **Best production quantity:** the corresponding feasible order quantity.
- **Average simulated profit:** the estimated expected profit for the selected decision.

Because the algorithm uses random samples, results may vary slightly between executions.

The solution is optimal within the evaluated price grid and the simulated scenarios; it is not necessarily the exact global optimum.


#### Implementation

The stochastic optimizer receives the demand parameters, production costs, quantity constraints, uncertainty rate, number of simulations, and price step as function arguments.

This design makes the optimization reusable with different input values without modifying the function's internal logic.

The optimizer is integrated into an interactive application that provides:

- A user input form with parameter validation.
- Deterministic and stochastic profit optimization.
- Monte Carlo risk analysis.
- Budget sensitivity analysis.
- A structured results dashboard.
- Visualizations of profitability, uncertainty, and budget constraints.

The interactive interface and its results are described in Sections 4.10–4.12. Model limitations and potential future improvements are discussed in Section 5.


In [21]:
def optimize_stochastic_profit(
    price_1, price_2, a, b,
    moq, max_quantity, fixed_cost, unit_cost,
    uncertainty_rate,
    num_simulations=10000,
    price_step=0.05
):
    
    min_price = min(price_1, price_2)
    max_price = max(price_1, price_2)

    
    # Generate unique candidate prices in integer cents
    min_cents = round(min_price * 100)
    max_cents = round(max_price * 100)
    step_cents = round(price_step * 100)

    price_cents = list(range(
        min_cents, max_cents + 1, step_cents
    ))

    # Always include the upper price boundary
    if price_cents[-1] != max_cents:
        price_cents.append(max_cents)

    prices = [cents / 100 for cents in price_cents]


    random_factors = np.random.normal(
        loc=0,
        scale=1,
        size=num_simulations
    )

    best_average_profit = float("-inf")
    best_stochastic_quantity = None
    best_stochastic_price = None

    for price in prices:
        demand_scenarios = (
            calculate_demand(price, a, b)
            + calculate_demand_std(price, a, b, uncertainty_rate) * random_factors
        )

        demand_scenarios = np.rint(demand_scenarios)
        demand_scenarios = np.maximum(0, demand_scenarios)

        for q in range(moq, max_quantity + 1):
            sales = np.minimum(q, demand_scenarios)
            profit = price * sales - calculate_total_cost(q, fixed_cost, unit_cost)
            average_profit = np.mean(profit)

            if average_profit > best_average_profit:
                best_average_profit = average_profit
                best_stochastic_quantity = q
                best_stochastic_price = price

    return best_stochastic_quantity, best_stochastic_price, best_average_profit


best_stochastic_quantity, best_stochastic_price, best_average_profit = (
    optimize_stochastic_profit(
    price_1, price_2, a, b,
    moq, max_quantity, fixed_cost, unit_cost,
    uncertainty_rate,
    num_simulations=5000,
    price_step=0.10
)
)

print(
    f"Best quantity: {best_stochastic_quantity}\n"
    f"Best price: {best_stochastic_price:.2f} EUR\n"
    f"Average simulated profit: {best_average_profit:.2f} EUR"
)

assert moq <= best_stochastic_quantity <= max_quantity
assert calculate_total_cost(best_stochastic_quantity, fixed_cost, unit_cost) <= budget


Best quantity: 1404
Best price: 4.40 EUR
Average simulated profit: 2830.74 EUR



### 4.10. Interactive User Interface

The Production Profit Optimizer includes an interactive user interface built with `ipywidgets`. It allows users to change the production parameters and run the optimization without modifying the Python code.

#### User Inputs

The interface organizes the input parameters into three main categories:

**1. Production Costs and Constraints**
- **Production Budget (EUR):** Maximum amount available for production.
- **Fixed Cost (EUR):** Costs that do not depend on production quantity.
- **Unit Cost (EUR):** Variable production cost per unit.
- **Minimum Order Quantity (MOQ):** Minimum number of units that must be produced.

**2. Price and Demand**
- **Price 1 and Demand 1:** First observation used to estimate the demand function.
- **Price 2 and Demand 2:** Second observation used to estimate the demand function.

These observations define the linear demand model:

\[
D(p) = a + bp
\]

where \(p\) is the selling price, \(a\) is the intercept, and \(b\) represents the change in demand per unit change in price.

**3. Simulation Settings**
- **Demand Uncertainty:** Relative standard deviation of simulated demand.
- **Number of Simulations:** Number of Monte Carlo scenarios used in the analysis.
- **Price Step:** Increment between candidate selling prices in stochastic optimization.

#### Input Validation

Before running the optimization, the application validates the input values to prevent invalid calculations.

The validation checks include:
- Positive production budget and unit cost.
- Non-negative fixed costs and demand observations.
- Positive minimum order quantity.
- Sufficient budget to produce the minimum required quantity.
- Two different positive price observations.
- A decreasing linear demand relationship.
- Valid simulation settings and uncertainty levels.
- Finite numerical values, excluding `NaN` and infinity.
- A computational workload limit to prevent excessively large simulations.

If an input is invalid, the application displays an error message instead of running the optimization.

#### Optimization Workflow

When the user clicks **Run Optimization**, the application:

1. Reads and validates the input parameters.
2. Calculates the linear demand model.
3. Determines the maximum affordable production quantity.
4. Finds the optimal selling price and production quantity under deterministic demand.
5. Performs stochastic optimization using Monte Carlo simulations.
6. Estimates the expected profit, probability of loss, and profit variability.
7. Evaluates how the maximum achievable profit changes with the available budget.
8. Displays the results in a structured dashboard with supporting visualizations.

This interface makes the mathematical model easier to explore and allows different production scenarios to be compared interactively.


In [22]:
# Common layout settings
widget_layout = widgets.Layout(width="220px")
label_layout = widgets.Layout(width="200px")

# Helper function for consistent input alignment
def create_input_row(label, widget):
    widget.description = ""

    return widgets.HBox([
        widgets.Label(label, layout=label_layout),
        widget
    ])


# Production costs and constraints
budget_widget = widgets.FloatText(
    value=5000.0,
    layout=widget_layout
)

fixed_cost_widget = widgets.FloatText(
    value=500.0,
    layout=widget_layout
)

unit_cost_widget = widgets.FloatText(
    value=1.50,
    layout=widget_layout
)

moq_widget = widgets.IntText(
    value=1000,
    layout=widget_layout
)


# Price and demand observations
price_1_widget = widgets.FloatText(
    value=3.0,
    layout=widget_layout
)

demand_1_widget = widgets.IntText(
    value=2000,
    layout=widget_layout
)

price_2_widget = widgets.FloatText(
    value=5.0,
    layout=widget_layout
)

demand_2_widget = widgets.IntText(
    value=1000,
    layout=widget_layout
)


# Stochastic simulation settings
uncertainty_widget = widgets.FloatSlider(
    value=0.20,
    min=0.0,
    max=1.0,
    step=0.05,
    readout_format=".0%",
    layout=widgets.Layout(width="290px")
)

simulations_widget = widgets.IntText(
    value=5000,
    layout=widget_layout
)

price_step_widget = widgets.FloatText(
    value=0.10,
    layout=widget_layout
)


# Display the complete input form
input_form = widgets.VBox([
    widgets.HTML("<h3>Production Costs and Constraints</h3>"),
    create_input_row("Budget (€):", budget_widget),
    create_input_row("Fixed Cost (€):", fixed_cost_widget),
    create_input_row("Unit Cost (€):", unit_cost_widget),
    create_input_row("Minimum Order Quantity:", moq_widget),

    widgets.HTML("<h3>Price and Demand Observations</h3>"),
    create_input_row("Price 1 (€):", price_1_widget),
    create_input_row("Demand 1 (units):", demand_1_widget),
    create_input_row("Price 2 (€):", price_2_widget),
    create_input_row("Demand 2 (units):", demand_2_widget),

    widgets.HTML("<h3>Stochastic Simulation Settings</h3>"),
    create_input_row("Uncertainty Rate:", uncertainty_widget),
    create_input_row("Simulations:", simulations_widget),
    create_input_row("Price Step (€):", price_step_widget)
])

display(input_form)


In [ ]:
run_button = widgets.Button(
    description="Run Optimization",
    button_style="success",
    icon="play",
    layout=widgets.Layout(width="200px")
)

output_widget = widgets.Output()

display(run_button, output_widget)

Button(button_style='success', description='Run Optimization', icon='play', layout=Layout(width='200px'), styl…

Output()

In [24]:
def validate_inputs(
    budget, fixed_cost, unit_cost, moq,
    price_1, demand_1, price_2, demand_2,
    uncertainty_rate, num_simulations, price_step
):

    # Reject NaN and infinite numeric inputs
    numeric_values = [
        budget, fixed_cost, unit_cost,
        moq, price_1, demand_1,
        price_2, demand_2,
        uncertainty_rate, num_simulations, price_step
    ]


    if not all(math.isfinite(value) for value in numeric_values):
        raise ValueError("All numeric inputs must be finite numbers.")
    
    if budget <= 0:
        raise ValueError("Budget must be greater than zero.")

    if fixed_cost < 0:
        raise ValueError("Fixed cost cannot be negative.")

    if unit_cost <= 0:
        raise ValueError("Unit cost must be greater than zero.")

    if moq <= 0:
        raise ValueError("Minimum order quantity must be greater than zero.")

    if fixed_cost + unit_cost * moq > budget:
        raise ValueError("Insufficient budget to meet the minimum order quantity.")

    if price_1 <= 0 or price_2 <= 0:
        raise ValueError("Selling prices must be greater than zero.")

    if price_1 == price_2:
        raise ValueError("The two selling prices must be different.")

    if demand_1 < 0 or demand_2 < 0:
        raise ValueError("Customer demand cannot be negative.")

    b = (demand_2 - demand_1) / (price_2 - price_1)

    if b >= 0:
        raise ValueError("Demand must decrease as selling price increases.")

    if not 0 <= uncertainty_rate <= 1:
        raise ValueError("Uncertainty rate must be between 0 and 1.")

    if num_simulations <= 0:
        raise ValueError("Number of simulations must be greater than zero.")

    # Validate price precision and search step
    if price_step < 0.01 or not math.isclose(
        price_step * 100, round(price_step * 100), abs_tol=1e-8
    ):
        raise ValueError(
            "Price step must be at least 0.01 EUR "
            "and a multiple of 0.01 EUR."
        )

    if any(
        not math.isclose(price * 100, round(price * 100), abs_tol=1e-8)
        for price in (price_1, price_2)
    ):
        raise ValueError(
            "Selling prices must have at most two decimal places."
        )


    max_quantity = int((budget - fixed_cost) // unit_cost)

    quantity_count = max_quantity - moq + 1

    min_price = min(price_1, price_2)
    max_price = max(price_1, price_2)

    
    # Count candidate prices without allocating a NumPy array
    min_cents = round(min_price * 100)
    max_cents = round(max_price * 100)
    step_cents = round(price_step * 100)

    price_range_cents = max_cents - min_cents

    price_count = price_range_cents // step_cents + 1

    # Include the upper boundary if the step does not reach it exactly
    if price_range_cents % step_cents != 0:
        price_count += 1


    total_evaluations = quantity_count * price_count * num_simulations

    max_evaluations = 250_000_000

    if total_evaluations > max_evaluations:
        raise ValueError(
            "The selected inputs require too many simulations. "
            "Please reduce the budget, number of simulations, "
            "or increase the price step."
        )

In [25]:
# Test with valid inputs
validate_inputs(
    5000, 500, 1.50, 1000,
    3.0, 2000, 5.0, 1000,
    0.20, 5000, 0.10
)

print("Valid input test passed!")

Valid input test passed!


In [26]:
# Test with insufficient budget
try:
    validate_inputs(
        1000, 500, 1.50, 1000,
        3.0, 2000, 5.0, 1000,
        0.20, 5000, 0.10
    )
    print("ERROR: Invalid inputs were accepted!")

except ValueError as error:
    print(f"Validation successful: {error}")

Validation successful: Insufficient budget to meet the minimum order quantity.


In [27]:
# Test input validation with NaN

try:
    validate_inputs(
        float("nan"), 500, 1.50, 1000,
        3.0, 2000, 5.0, 1000,
        0.20, 5000, 0.10
    )
    print("ERROR: NaN was accepted!")

except ValueError as error:
    print(f"Validation successful: {error}")

Validation successful: All numeric inputs must be finite numbers.


In [28]:
# Test input validation with Infinity

try:
    validate_inputs(
        float("inf"), 500, 1.50, 1000,
        3.0, 2000, 5.0, 1000,
        0.20, 5000, 0.10
    )
    print("ERROR: Infinity was accepted!")

except ValueError as error:
    print(f"Validation successful: {error}")

Validation successful: All numeric inputs must be finite numbers.


In [29]:
def calculate_profit_curve(
    price_1, price_2, a, b,
    moq, max_quantity, fixed_cost, unit_cost
):
    prices = []
    profits = []

    min_price = min(price_1, price_2)
    max_price = max(price_1, price_2)

    for price_cents in range(round(min_price * 100), round(max_price * 100) + 1):
        price = price_cents / 100

        quantity = calculate_optimal_quantity(
            price, a, b, moq, max_quantity, fixed_cost, unit_cost
        )

        profit = calculate_profit(
            quantity, price, fixed_cost, unit_cost, a, b
        )

        prices.append(price)
        profits.append(profit)

    return prices, profits

In [30]:
prices, profits = calculate_profit_curve(
    price_1, price_2, a, b,
    moq, max_quantity, fixed_cost, unit_cost
)

print(f"Number of evaluated prices: {len(prices)}")
print(f"Maximum profit: {max(profits):.2f} EUR")
print(f"Price at maximum profit: {prices[profits.index(max(profits))]:.2f} EUR")

Number of evaluated prices: 201
Maximum profit: 3281.25 EUR
Price at maximum profit: 4.25 EUR


In [31]:
def analyze_budget_sensitivity(
    budget_values, fixed_cost, unit_cost, moq,
    price_1, price_2, a, b
):
    results = []

    for current_budget in budget_values:
        max_quantity = int((current_budget - fixed_cost) // unit_cost)

        if max_quantity < moq:
            results.append(None)
            continue

        best_profit = float("-inf")

        min_price = min(price_1, price_2)
        max_price = max(price_1, price_2)

        for price_cents in range(round(min_price * 100), round(max_price * 100) + 1):
            price = price_cents / 100

            quantity = calculate_optimal_quantity(
                price, a, b, moq, max_quantity, fixed_cost, unit_cost
            )

            profit = calculate_profit(
                quantity, price, fixed_cost, unit_cost, a, b
            )

            if profit > best_profit:
                best_profit = profit

        results.append(best_profit)

    return results

In [32]:
budget_values = [1500, 2000, 2500, 3000, 4000, 5000, 6000]

sensitivity_profits = analyze_budget_sensitivity(
    budget_values,
    fixed_cost, unit_cost, moq,
    price_1, price_2, a, b
)

for current_budget, profit in zip(budget_values, sensitivity_profits):
    if profit is None:
        print(f"Budget: {current_budget} EUR -> Infeasible")
    else:
        print(f"Budget: {current_budget} EUR -> Maximum Profit: {profit:.2f} EUR")

Budget: 1500 EUR -> Infeasible
Budget: 2000 EUR -> Maximum Profit: 3000.00 EUR
Budget: 2500 EUR -> Maximum Profit: 3277.20 EUR
Budget: 3000 EUR -> Maximum Profit: 3281.25 EUR
Budget: 4000 EUR -> Maximum Profit: 3281.25 EUR
Budget: 5000 EUR -> Maximum Profit: 3281.25 EUR
Budget: 6000 EUR -> Maximum Profit: 3281.25 EUR


In [33]:
def show_results_dashboard(
    a, b, max_quantity,
    best_price, best_quantity, best_profit,
    best_stochastic_price, best_stochastic_quantity, best_average_profit,
    num_simulations, average_profit, loss_probability, profit_std
):
    html = f"""
    <div style="
        font-family: Arial, sans-serif;
        background: #ffffff;
        border: 1px solid #d0d7de;
        border-radius: 14px;
        padding: 20px;
        margin-bottom: 20px;
        box-shadow: 0 2px 8px rgba(0,0,0,0.06);
    ">
        <h2 style="margin-top: 0; margin-bottom: 18px; color: #1f2937;">
            Production Profit Optimizer Results
        </h2>

        <div style="
            display: flex;
            flex-wrap: wrap;
            gap: 16px;
        ">
            <!-- Deterministic card -->
            <div style="
                flex: 1 1 300px;
                background: #f8fafc;
                border: 1px solid #e5e7eb;
                border-radius: 12px;
                padding: 16px;
            ">
                <h3 style="margin-top: 0; color: #2563eb;">
                    Deterministic Optimization
                </h3>
                <table style="width: 100%; border-collapse: collapse; font-size: 14px;">
                    <tr>
                        <td style="padding: 6px 0; font-weight: bold;">Demand model</td>
                        <td style="padding: 6px 0;">D(p) = {a:.2f} + ({b:.2f}) × p</td>
                    </tr>
                    <tr>
                        <td style="padding: 6px 0; font-weight: bold;">Max quantity</td>
                        <td style="padding: 6px 0;">{max_quantity} units</td>
                    </tr>
                    <tr>
                        <td style="padding: 6px 0; font-weight: bold;">Optimal price</td>
                        <td style="padding: 6px 0;">{best_price:.2f} EUR</td>
                    </tr>
                    <tr>
                        <td style="padding: 6px 0; font-weight: bold;">Optimal quantity</td>
                        <td style="padding: 6px 0;">{best_quantity} units</td>
                    </tr>
                    <tr>
                        <td style="padding: 6px 0; font-weight: bold;">Maximum profit</td>
                        <td style="padding: 6px 0; color: #065f46; font-weight: bold;">
                            {best_profit:.2f} EUR
                        </td>
                    </tr>
                </table>
            </div>

            <!-- Stochastic card -->
            <div style="
                flex: 1 1 300px;
                background: #f8fafc;
                border: 1px solid #e5e7eb;
                border-radius: 12px;
                padding: 16px;
            ">
                <h3 style="margin-top: 0; color: #7c3aed;">
                    Stochastic Optimization
                </h3>
                <table style="width: 100%; border-collapse: collapse; font-size: 14px;">
                    <tr>
                        <td style="padding: 6px 0; font-weight: bold;">Optimal price</td>
                        <td style="padding: 6px 0;">{best_stochastic_price:.2f} EUR</td>
                    </tr>
                    <tr>
                        <td style="padding: 6px 0; font-weight: bold;">Optimal quantity</td>
                        <td style="padding: 6px 0;">{best_stochastic_quantity} units</td>
                    </tr>
                    <tr>
                        <td style="padding: 6px 0; font-weight: bold;">Average simulated profit</td>
                        <td style="padding: 6px 0; color: #065f46; font-weight: bold;">
                            {best_average_profit:.2f} EUR
                        </td>
                    </tr>
                </table>
            </div>

            <!-- Risk card -->
            <div style="
                flex: 1 1 300px;
                background: #f8fafc;
                border: 1px solid #e5e7eb;
                border-radius: 12px;
                padding: 16px;
            ">
                <h3 style="margin-top: 0; color: #dc2626;">
                    Monte Carlo Risk Analysis
                </h3>
                <table style="width: 100%; border-collapse: collapse; font-size: 14px;">
                    <tr>
                        <td style="padding: 6px 0; font-weight: bold;">Simulations</td>
                        <td style="padding: 6px 0;">{num_simulations}</td>
                    </tr>
                    <tr>
                        <td style="padding: 6px 0; font-weight: bold;">Average profit</td>
                        <td style="padding: 6px 0;">{average_profit:.2f} EUR</td>
                    </tr>
                    <tr>
                        <td style="padding: 6px 0; font-weight: bold;">Probability of loss</td>
                        <td style="padding: 6px 0;">{loss_probability * 100:.2f}%</td>
                    </tr>
                    <tr>
                        <td style="padding: 6px 0; font-weight: bold;">Profit standard deviation</td>
                        <td style="padding: 6px 0;">{profit_std:.2f} EUR</td>
                    </tr>
                </table>
            </div>
        </div>
    </div>
    """

    display(HTML(html))

In [34]:

def run_optimization(button):
    # Read input values from the interactive form
    budget = budget_widget.value
    fixed_cost = fixed_cost_widget.value
    unit_cost = unit_cost_widget.value
    moq = moq_widget.value

    price_1 = price_1_widget.value
    demand_1 = demand_1_widget.value
    price_2 = price_2_widget.value
    demand_2 = demand_2_widget.value

    uncertainty_rate = uncertainty_widget.value
    num_simulations = simulations_widget.value
    price_step = price_step_widget.value

    # Validate user inputs before running the optimization
    try:
        validate_inputs(
            budget, fixed_cost, unit_cost, moq,
            price_1, demand_1, price_2, demand_2,
            uncertainty_rate, num_simulations, price_step
        )

    except ValueError as error:
        with output_widget:
            output_widget.clear_output()
            print(f"Input error: {error}")
        return

    # Calculate linear demand model coefficients
    b = (demand_2 - demand_1) / (price_2 - price_1)
    a = demand_1 - b * price_1

    # Calculate maximum affordable production quantity
    max_quantity = int((budget - fixed_cost) // unit_cost)

    # Perform budget sensitivity analysis
    minimum_budget = fixed_cost + unit_cost * moq

    sensitivity_budgets = np.linspace(
        minimum_budget,
        max(budget, minimum_budget * 1.5),
        10
    )

    sensitivity_profits = analyze_budget_sensitivity(
        sensitivity_budgets,
        fixed_cost, unit_cost, moq,
        price_1, price_2, a, b
    )

    # Find the optimal deterministic price and quantity
    best_profit = float("-inf")
    best_price = None
    best_quantity = None

    min_price = min(price_1, price_2)
    max_price = max(price_1, price_2)

    for price_cents in range(
        round(min_price * 100),
        round(max_price * 100) + 1
    ):
        price = price_cents / 100

        optimal_quantity = calculate_optimal_quantity(
            price, a, b, moq, max_quantity, fixed_cost, unit_cost
        )

        profit = calculate_profit(
            optimal_quantity, price, fixed_cost, unit_cost, a, b
        )

        if profit > best_profit:
            best_profit = profit
            best_price = price
            best_quantity = optimal_quantity

    # Prepare deterministic profit curve data
    curve_prices, curve_profits = calculate_profit_curve(
        price_1, price_2, a, b,
        moq, max_quantity, fixed_cost, unit_cost
    )

    # Find the optimal stochastic price and quantity
    best_stochastic_quantity, best_stochastic_price, best_average_profit = (
        optimize_stochastic_profit(
            price_1, price_2, a, b,
            moq, max_quantity, fixed_cost, unit_cost,
            uncertainty_rate,
            num_simulations=num_simulations,
            price_step=price_step
        )
    )

    # Simulate profits for the selected stochastic strategy
    stochastic_profits = monte_carlo_simulation(
        best_stochastic_quantity,
        best_stochastic_price,
        num_simulations,
        fixed_cost,
        unit_cost,
        a,
        b,
        uncertainty_rate
    )

    # Calculate Monte Carlo risk metrics
    average_profit, loss_probability, profit_std = analyze_simulation(
        stochastic_profits
    )
    
    # Display optimization results and visualizations
    with output_widget:
        output_widget.clear_output()

        # Display results dashboard
        show_results_dashboard(
            a, b, max_quantity,
            best_price, best_quantity, best_profit,
            best_stochastic_price, best_stochastic_quantity, best_average_profit,
            len(stochastic_profits), average_profit, loss_probability, profit_std
        )

        # Visualize deterministic profit versus selling price
        plt.figure(figsize=(14, 5))

        plt.plot(
            curve_prices,
            curve_profits,
            label="Maximum Profit"
        )

        plt.scatter(
            best_price,
            best_profit,
            label="Optimal Solution",
            zorder=3
        )

        plt.title("Deterministic Profit vs. Selling Price")
        plt.xlabel("Selling Price (EUR)")
        plt.ylabel("Profit (EUR)")
        plt.legend()
        plt.grid(True)
        plt.show()

        # Visualize the Monte Carlo profit distribution
        plt.figure(figsize=(14, 5))

        plt.hist(
            stochastic_profits,
            bins=40,
            edgecolor="black",
            alpha=0.7
        )

        plt.axvline(
            average_profit,
            color="red",
            linestyle="--",
            label=f"Average Profit: {average_profit:.2f} EUR"
        )

        plt.title("Monte Carlo Profit Distribution")
        plt.xlabel("Simulated Profit (EUR)")
        plt.ylabel("Frequency")
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.show()

        # Visualize sensitivity of maximum profit to production budget
        plt.figure(figsize=(14, 5))

        plt.plot(
            sensitivity_budgets,
            sensitivity_profits,
            marker="o",
            label="Maximum Profit"
        )

        plt.axvline(
            budget,
            color="red",
            linestyle="--",
            label=f"Current Budget: {budget:.2f} EUR"
        )

        plt.title("Maximum Profit vs. Production Budget")
        plt.xlabel("Production Budget (EUR)")
        plt.ylabel("Maximum Profit (EUR)")
        plt.legend()
        plt.grid(True)
        plt.show()



### 4.11. Results Dashboard and Visualizations

The application presents the optimization results in a structured dashboard, followed by three visualizations that help users understand profitability, uncertainty, and budget constraints.

#### Results Dashboard

The dashboard contains three sections:

**1. Deterministic Optimization**

Displays the linear demand model, maximum affordable production quantity, optimal selling price, optimal production quantity, and maximum profit.

These results assume that actual demand is equal to the predicted demand.

**2. Stochastic Optimization**

Displays the selling price and production quantity that maximize estimated average profit under uncertain demand.

Unlike deterministic optimization, this approach considers multiple possible demand scenarios.

**3. Monte Carlo Risk Analysis**

Evaluates the selected stochastic strategy using simulated demand and reports:

- **Average Profit:** Mean profit across the simulated scenarios.
- **Probability of Loss:** Percentage of simulations with negative profit.
- **Profit Standard Deviation:** Measure of how much simulated profits vary around their mean.

Because Monte Carlo simulations involve randomness, results may vary slightly between runs.

#### Visualization 1: Deterministic Profit vs. Selling Price

This graph illustrates how the maximum achievable deterministic profit changes with selling price.

The highlighted point identifies the optimal price and its corresponding maximum profit.

#### Visualization 2: Monte Carlo Profit Distribution

The histogram shows the distribution of simulated profits for the selected stochastic strategy.

The dashed red line marks the average simulated profit.

The graph helps users identify the spread of possible outcomes and the presence of unfavorable scenarios.

#### Visualization 3: Maximum Profit vs. Production Budget

This graph shows how the maximum deterministic profit changes as the available production budget increases.

It helps identify situations where additional budget improves profitability and situations where increasing the budget no longer provides a meaningful benefit.

#### Interpretation of the Results

The deterministic solution represents the optimal decision when demand is predictable.

The stochastic solution considers demand fluctuations and aims to maximize average simulated profit.

Together, the dashboard and visualizations provide a more complete view of production profitability and risk.



### 4.12. Budget Sensitivity Analysis

Budget sensitivity analysis examines how changes in the available production budget affect the maximum achievable profit.

The analysis uses the deterministic profit model and recalculates the optimal selling price and production quantity for different budget levels.

#### Methodology

For each budget value, the model:

1. Calculates the maximum affordable production quantity.
2. Checks whether the budget is sufficient to meet the minimum order quantity (MOQ).
3. Evaluates possible selling prices within the specified price range.
4. Determines the maximum achievable profit under the corresponding budget constraint.

The maximum affordable production quantity is:

\[
Q_{\max} = \left\lfloor \frac{B-F}{c} \right\rfloor
\]

where:

- \(B\) = available production budget;
- \(F\) = fixed production cost;
- \(c\) = variable cost per unit.

If the available budget is lower than the cost of producing the minimum order quantity, the scenario is considered infeasible.

#### Example Results

Using the default production parameters:

| Production Budget (EUR) | Maximum Profit (EUR) |
|---|---|
| 1,500 | Infeasible |
| 2,000 | 3,000.00 |
| 2,500 | 3,277.20 |
| 3,000 | 3,281.25 |
| 4,000 | 3,281.25 |
| 5,000 | 3,281.25 |
| 6,000 | 3,281.25 |

#### Interpretation

At a budget of 1,500 EUR, the business cannot afford the minimum required production quantity.

Increasing the budget initially allows the business to produce more units and achieve higher profit.

However, the maximum profit reaches approximately 3,281.25 EUR once the budget is sufficient to support the optimal production quantity.

Further budget increases do not improve the deterministic maximum profit because additional production is not justified by the predicted market demand.

#### Business Implications

The analysis demonstrates that the optimal production decision depends on both financial constraints and customer demand.

A larger budget increases production capacity, but it does not guarantee higher profitability.

Businesses should therefore consider demand forecasts and production costs before committing additional financial resources.



### 4.13. Testing and Validation

Testing was performed to verify the mathematical correctness, reliability, and expected behavior of the Production Profit Optimizer under different input conditions.

#### 1. Mathematical Model Validation

The deterministic optimization results were compared with an analytical solution obtained using derivatives.

For the default parameters, the model produces:

- Optimal selling price: **4.25 EUR**
- Optimal production quantity: **1,375 units**
- Maximum deterministic profit: **3,281.25 EUR**

The numerical optimization agrees with the analytical result.

#### 2. Deterministic and Stochastic Consistency

The optimization algorithms were tested with zero demand uncertainty.

When uncertainty is set to zero, simulated demand becomes deterministic.

With a sufficiently fine price step, the stochastic optimizer produces the same optimal price, production quantity, and profit as the deterministic optimizer.

This confirms that both algorithms are consistent when demand uncertainty is removed.

#### 3. Budget Constraint Testing

Different production budgets were evaluated to verify that the optimizer respects financial constraints.

The tests included:

- Budgets insufficient to meet the minimum order quantity.
- Budgets that restrict the maximum affordable production quantity.
- Budgets sufficient to achieve the unconstrained optimal production quantity.

The model correctly identifies infeasible scenarios and respects production limits.

#### 4. Input Validation Testing

The input validation function was tested with invalid and boundary values, including:

- Non-positive production budgets and unit costs.
- Negative fixed costs and demand observations.
- Invalid minimum order quantities.
- Identical price observations.
- Non-decreasing demand relationships.
- Invalid simulation settings.
- `NaN` and infinite numerical values.
- Computational workloads exceeding the configured safety limit.

Invalid inputs are rejected with descriptive error messages.

#### 5. Monte Carlo Simulation Testing

The stochastic model was evaluated under different levels of demand uncertainty.

The simulation produces a distribution of possible profits and calculates the following risk measures:

- Average simulated profit.
- Probability of financial loss.
- Standard deviation of simulated profit.

Results vary between runs because of random demand generation.

#### 6. Interactive Interface Testing

The interactive interface was tested by changing production inputs and running the optimization.

The results dashboard and visualizations update according to the selected parameters.

Invalid inputs are handled through validation messages rather than allowing the optimization to proceed.

#### Testing Summary

The completed tests support the correctness of the deterministic calculations, the consistency of the stochastic model, and the handling of the tested input conditions.

Monte Carlo results remain estimates and depend on the selected demand distribution, uncertainty level, and number of simulations.



## 5. Conclusions and Limitations

### 5.1. Project Conclusions

The Production Profit Optimizer demonstrates how mathematical modeling and Python can support business decisions related to production planning, pricing, and profitability.

The project combines deterministic optimization, stochastic optimization, Monte Carlo simulations, and budget sensitivity analysis.

Using the default input parameters, the deterministic model identifies:

- **Optimal selling price:** 4.25 EUR
- **Optimal production quantity:** 1,375 units
- **Maximum deterministic profit:** 3,281.25 EUR

The stochastic model extends this approach by considering fluctuations in customer demand and selecting a production strategy that maximizes estimated average profit.

Monte Carlo simulations provide additional information about possible financial outcomes, including expected profit, probability of loss, and profit variability.

Budget sensitivity analysis demonstrates that increasing available funds does not necessarily increase profitability. Once the budget is sufficient to support the optimal production quantity, additional production capacity may provide no benefit under the assumed demand model.

Overall, the project illustrates the importance of considering market demand, production costs, financial constraints, and uncertainty when making production decisions.

### 5.2. Model Limitations

Despite its practical applications, the model relies on several simplifying assumptions.

**Linear Demand Model**

Customer demand is assumed to follow a linear relationship with selling price, estimated from two observations. Real-world demand may be nonlinear and influenced by competitors, seasonality, marketing, and other factors.

**Simplified Cost Structure**

The model considers fixed costs and a constant production cost per unit. It does not account for quantity discounts, storage costs, transportation, or other variable business expenses.

**Demand Uncertainty**

Stochastic demand is modeled using a normal distribution, with simulated demand constrained to non-negative integer values. Actual demand may follow a different probability distribution.

**Single-Product Optimization**

The model optimizes production and pricing for one product. It does not consider shared resources, multiple products, or interactions between different product categories.

**Simulation-Based Estimates**

Stochastic optimization and risk metrics are based on Monte Carlo simulations. Their results depend on the simulation count and random demand scenarios, so repeated runs may produce slightly different estimates.

**Discrete Search**

The optimization searches a defined range of selling prices and integer production quantities. The quality of the numerical solution depends partly on the selected price increments.

### 5.3. Potential Future Improvements

Possible extensions include:

- Using historical sales data to estimate more realistic demand functions.
- Supporting nonlinear demand models.
- Introducing additional cost categories and production constraints.
- Extending optimization to multiple products.
- Comparing alternative probability distributions for uncertain demand.
- Improving computational performance for larger simulation scenarios.
- Adding export functionality for optimization results.

### 5.4. Final Summary

The Production Profit Optimizer provides an interactive decision-support tool that combines mathematical optimization with uncertainty analysis.

It demonstrates how concepts from algebra, calculus, probability, and numerical computation can be applied to a practical business problem using Python.


In [35]:
run_button.on_click(run_optimization)


## 6. References

### Course Materials

- SoftUni. *Math Concepts for Developers*, AI and Machine Learning Program, 2026. Course materials covering algebra, calculus, probability, and mathematical optimization.

### Technical Documentation

- NumPy Developers. *NumPy Documentation*. https://numpy.org/doc/stable/
- Matplotlib Development Team. *Matplotlib Documentation*. https://matplotlib.org/stable/
- Jupyter Development Team. *ipywidgets Documentation*. https://ipywidgets.readthedocs.io/en/stable/
- Python Software Foundation. *Python Documentation*. https://docs.python.org/3/

### Mathematical Model

The production profit model, optimization examples, calculations, simulations, and analyses presented in this project were developed as part of the coursework.

The mathematical explanations are based on concepts covered in the course. The Python libraries listed above are used to implement the numerical computations, interactive interface, and visualizations.
